In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

In [2]:
from ast import literal_eval

df = pd.read_csv('data/matches_processed.csv')
df["Player"] = df["Player"].apply(literal_eval)

In [3]:
import torch
from torch.utils.data import Dataset
from sklearn.preprocessing import LabelEncoder

class WinPredictionDataset(Dataset):
    def __init__(self, players, result):
        self.players = players
        self.results = torch.tensor(result, dtype=torch.float32)
        
        # Flatten all tokens to build vocabulary
        all_tokens = [item for row in players for subsublist in row for item in subsublist]
        self.tokenizer = LabelEncoder()
        self.tokenizer.fit(all_tokens)  # Fit on all possible tokens
        
        # Pre-encode all data during init (more efficient)
        self.encoded_players = [
            [
                self.tokenizer.transform(subsublist) 
                for subsublist in row
            ] 
            for row in players
        ]
        
    def __len__(self):
        return len(self.players)

    def __getitem__(self, idx):
        return {
            "players": torch.tensor(np.array(self.encoded_players[idx]), dtype=torch.long),  # Shape: [10, 5]
            "results": self.results[idx]  # Shape: [1]
        }

In [ ]:
import torch
import torch.nn as nn
import math

class SinusoidalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 50):  # 10×5=50
        super().__init__()
        position = torch.arange(max_len).unsqueeze(1)  # [0, 1, ..., 49]
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)  # Even dims: sine
        pe[:, 1::2] = torch.cos(position * div_term)  # Odd dims: cosine
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:x.size(1)] 

class WinPredictionModel(nn.Module):
    def __init__(self, vocab_size=511, embedding_dim=64):
        super().__init__()
        # (1) Stat embeddings (from token IDs)
        self.stat_embedding = nn.Embedding(vocab_size, embedding_dim)
        
        # (2) LEARNABLE role embeddings (5 roles: Top, Jungle, Mid, etc.)
        self.role_embedding = nn.Embedding(5, embedding_dim)  # 5 roles
        
        # (3) Team embeddings (allied=0, enemy=1)
        self.team_embedding = nn.Embedding(2, embedding_dim)
        
        # (4) CNN layers
        self.conv = nn.Sequential(
            nn.Conv2d(embedding_dim, 128, kernel_size=(5, 5), padding=(2, 2)), # [batch, 128, 10, 5]
            nn.BatchNorm2d(128),  # Stabilize training
            nn.ReLU(),
            
            nn.Conv2d(128, 128, kernel_size=(5, 5), padding=(2, 2)),  # [batch, 128, 10, 5]
            nn.BatchNorm2d(128),  # Stabilize training
            nn.ReLU()
        )
        
        # (5) New Positional Encoding + Transformer encoder
        self.pos_encoder = SinusoidalEncoding(d_model=128)
        
        self.encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=128, nhead=8, dim_feedforward=512, dropout=0.5, activation='gelu', batch_first=True
            ),
            num_layers=3,
            enable_nested_tensor=True 
        )
        
        # (6) Final classifier
        # Classifier
        self.head = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        # x: [batch, 10 players, 5 stats]
        batch_size = x.size(0)
        
        # (1) Embed stats -> [batch, 10, 5, 64]
        embedded = self.stat_embedding(x)
        
        # (2) Add LEARNABLE role embeddings
        role_ids = torch.arange(5).repeat(2).to(x.device)  # [0,1,2,3,4, 0,1,2,3,4] (10 players)
        role_embedded = self.role_embedding(role_ids)      # [10, 64]
        embedded += role_embedded.unsqueeze(0).unsqueeze(2)  # [batch, 10, 5, 64]
        
        # (3) Add team embeddings
        team_ids = torch.cat([
            torch.zeros(batch_size, 5, dtype=torch.long),  # allied
            torch.ones(batch_size, 5, dtype=torch.long)    # enemy
        ], dim=1).to(x.device)
        team_embedded = self.team_embedding(team_ids).unsqueeze(2)  # [batch, 10, 1, 64]
        embedded += team_embedded
        
        # Reshape for CNN: [batch, 64, 10, 5]
        embedded = embedded.permute(0, 3, 1, 2)
        
        # (4) Apply CNN layers
        x = self.conv(embedded) # [batch, 128, 10, 5]
        
        # Update x shape since transformer encoder expects a sequence
        x = x.permute(0, 2, 3, 1)  # [batch, 10, 5, 128]
        x = x.reshape(x.shape[0], x.shape[1] * x.shape[2], x.shape[3])  # [batch, 50, 128]
        
        # (5) Add one more time positional encodings and pass through transformer encoder
        x = self.pos_encoder(x)
        x = self.encoder(x)
        
        # (6) Global average pooling and pass through classifier
        x = x.mean(dim=1)
        return self.head(x)
        
        

In [5]:
from sklearn.metrics import confusion_matrix, accuracy_score

def eval_model(model, val_loader, criterion, device):
    model.eval()
    
    all_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch in val_loader:
            players = batch["players"].to(device)
            labels = batch["results"].to(device)
            
            outputs = model(players)
            loss = criterion(outputs, labels.unsqueeze(1)) 
            all_loss += loss.item()
            
            preds = np.round(outputs.cpu().numpy())
            
            all_preds.extend(preds.flatten().tolist())
            all_labels.extend(labels.cpu().numpy().flatten().tolist())
            
    average_loss = all_loss / len(val_loader)            
    accuracy = accuracy_score(all_labels, all_preds)
    conf_matrix = confusion_matrix(all_labels, all_preds)  # Call the function from sklearn.metrics
    return average_loss, accuracy, conf_matrix

In [6]:
from torch.utils.data import DataLoader
from sklearn.model_selection import KFold
import torch
import numpy as np
import visualize

torch.manual_seed(42)
np.random.seed(42)

n_folds = 5
num_epochs = 20
learning_rate = 2e-4
batch_size = 128

dataset = WinPredictionDataset(df["Player"].values, df["Win"].to_numpy())

kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

fold_train_losses = []
fold_val_losses = []
fold_accuracies = []
fold_conf_matrices = []

for fold, (train_idx, val_idx) in enumerate(kf.split(dataset)):
    print(f"Fold {fold + 1}/{kf.n_splits}")
    
    trainset = torch.utils.data.Subset(dataset, train_idx)
    valset = torch.utils.data.Subset(dataset, val_idx)

    train_loader = DataLoader(trainset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(valset, batch_size=batch_size, shuffle=False)
    
    # Initialize model, optimizer, and loss function
    model = WinPredictionModel(vocab_size=dataset.tokenizer.classes_.size).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.BCELoss()

    train_losses = []
    val_losses = []
    accuracies = []
    conf_matrices = []

    for epoch in range(num_epochs):
        train_loss = 0
        model.train()
        for batch in train_loader:
            players = batch["players"].to(device)
            results = batch["results"].to(device)
            
            optimizer.zero_grad()
            outputs = model(players)
            loss = criterion(outputs, results.unsqueeze(1))  # Ensure results are the same shape as outputs
            
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            
        average_train_loss = train_loss / len(train_loader)
        average_val_loss, accuracy, conf_matrix = eval_model(model, val_loader, criterion, device)
        
        print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {average_train_loss}/{average_val_loss}, Validation accuracy: {accuracy}")
        
        train_losses.append(average_train_loss)
        val_losses.append(average_val_loss)
        accuracies.append(accuracy)
        
    fold_train_losses.append(train_losses)
    fold_val_losses.append(val_losses)
    fold_accuracies.append(accuracies)
    fold_conf_matrices.append(conf_matrix)
    
# Average the results across folds
train_losses = np.mean(fold_train_losses, axis=0)
val_losses = np.mean(fold_val_losses, axis=0)
accuracies = np.mean(fold_accuracies, axis=0)
conf_matrices = np.mean(fold_conf_matrices, axis=0)

visualize.loss(train_losses, val_loss=val_losses, title=f"Train and Validation Losses across {n_folds} folds")
visualize.accuracy(accuracies, title=f"Validation Accuracy across {n_folds} folds")
visualize.confusion_matrix(conf_matrices, title=f"Confusion Matrix across {n_folds} folds")

Using device: cuda
Fold 1/5


c:\Users\jager\Desktop\github\win_prediction\venv\Lib\site-packages\torch\nn\modules\transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


Epoch 1/20, Loss: 0.6927022385218787/0.6926214396953583, Validation accuracy: 0.51
Epoch 2/20, Loss: 0.6902389053314452/0.6938468404114246, Validation accuracy: 0.488
Epoch 3/20, Loss: 0.6845255844176762/0.697440892457962, Validation accuracy: 0.5045
Epoch 4/20, Loss: 0.6801331639289856/0.6886553019285202, Validation accuracy: 0.5345
Epoch 5/20, Loss: 0.669705501624516/0.6934780701994896, Validation accuracy: 0.541
Epoch 6/20, Loss: 0.6598154883536081/0.6913102567195892, Validation accuracy: 0.5345
Epoch 7/20, Loss: 0.6497463300114587/0.7093940041959286, Validation accuracy: 0.5355
Epoch 8/20, Loss: 0.6433362430996366/0.6940523646771908, Validation accuracy: 0.539
Epoch 9/20, Loss: 0.6250161481282067/0.738996759057045, Validation accuracy: 0.543
Epoch 10/20, Loss: 0.6099002143693348/0.7327015176415443, Validation accuracy: 0.541
Epoch 11/20, Loss: 0.6032497182724967/0.7560721710324287, Validation accuracy: 0.542
Epoch 12/20, Loss: 0.5803356151732187/0.7482095137238503, Validation accur

c:\Users\jager\Desktop\github\win_prediction\venv\Lib\site-packages\torch\nn\modules\transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


Epoch 1/20, Loss: 0.6930701231199597/0.6927791461348534, Validation accuracy: 0.51
Epoch 2/20, Loss: 0.6906656783724588/0.6947727054357529, Validation accuracy: 0.51
Epoch 3/20, Loss: 0.685170330698528/0.687882374972105, Validation accuracy: 0.528
Epoch 4/20, Loss: 0.6763741620003231/0.681453388184309, Validation accuracy: 0.546
Epoch 5/20, Loss: 0.6644778242186894/0.68148148432374, Validation accuracy: 0.5575
Epoch 6/20, Loss: 0.6581731069655645/0.6833550371229649, Validation accuracy: 0.554
Epoch 7/20, Loss: 0.6495183252152943/0.7011221349239349, Validation accuracy: 0.545
Epoch 8/20, Loss: 0.6405774033258832/0.6917205266654491, Validation accuracy: 0.5605
Epoch 9/20, Loss: 0.6307787734364706/0.707115713506937, Validation accuracy: 0.555
Epoch 10/20, Loss: 0.6189969919976734/0.696969885379076, Validation accuracy: 0.563
Epoch 11/20, Loss: 0.615474990436009/0.7111028879880905, Validation accuracy: 0.5495
Epoch 12/20, Loss: 0.5963530067413573/0.7230033427476883, Validation accuracy: 0.

...

Epoch 1/20, Loss: 0.6936117467426118/0.6914195194840431, Validation accuracy: 0.5295
Epoch 2/20, Loss: 0.6923957391390725/0.6905679255723953, Validation accuracy: 0.5295
Epoch 3/20, Loss: 0.689611189895206/0.6900199353694916, Validation accuracy: 0.5295
Epoch 4/20, Loss: 0.6810379028320312/0.6881196014583111, Validation accuracy: 0.5265
Epoch 5/20, Loss: 0.6709902409523253/0.683677714318037, Validation accuracy: 0.5495
Epoch 6/20, Loss: 0.6611048645443387/0.6802482195198536, Validation accuracy: 0.5545
Epoch 7/20, Loss: 0.653361145466093/0.6851811222732067, Validation accuracy: 0.5495
Epoch 8/20, Loss: 0.6435524499605573/0.7017698958516121, Validation accuracy: 0.5265
Epoch 9/20, Loss: 0.635950367602091/0.6949018090963364, Validation accuracy: 0.5475
Epoch 10/20, Loss: 0.6239116854137845/0.7182451896369457, Validation accuracy: 0.539
Epoch 11/20, Loss: 0.6127914587656657/0.731377612799406, Validation accuracy: 0.541
Epoch 12/20, Loss: 0.5971900101691957/0.7351723089814186, Validation a

c:\Users\jager\Desktop\github\win_prediction\venv\Lib\site-packages\torch\nn\modules\transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


Epoch 1/20, Loss: 0.6934445708517044/0.6903553269803524, Validation accuracy: 0.531
Epoch 2/20, Loss: 0.6923435396618314/0.6883504390716553, Validation accuracy: 0.5245
Epoch 3/20, Loss: 0.6866195466783311/0.68186105042696, Validation accuracy: 0.5545
Epoch 4/20, Loss: 0.6785935617628551/0.6805156208574772, Validation accuracy: 0.5615
Epoch 5/20, Loss: 0.6676628759929112/0.6744017079472542, Validation accuracy: 0.57
Epoch 6/20, Loss: 0.6544074567537459/0.6841712221503258, Validation accuracy: 0.5655
Epoch 7/20, Loss: 0.6451854308446249/0.6901484690606594, Validation accuracy: 0.574
Epoch 8/20, Loss: 0.6356056663725111/0.6902066022157669, Validation accuracy: 0.5565
Epoch 9/20, Loss: 0.6212426130733792/0.6974727250635624, Validation accuracy: 0.5565
Epoch 10/20, Loss: 0.6020429011375185/0.6982378698885441, Validation accuracy: 0.5735
Epoch 11/20, Loss: 0.5906160256219288/0.7471701428294182, Validation accuracy: 0.544
Epoch 12/20, Loss: 0.5746149222056071/0.7399449832737446, Validation a

c:\Users\jager\Desktop\github\win_prediction\venv\Lib\site-packages\torch\nn\modules\transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


Epoch 1/20, Loss: 0.6930619487686763/0.6903711557388306, Validation accuracy: 0.539
Epoch 2/20, Loss: 0.68936535574141/0.688479658216238, Validation accuracy: 0.541
Epoch 3/20, Loss: 0.6829694651422047/0.6845018193125725, Validation accuracy: 0.5475
Epoch 4/20, Loss: 0.674849494109078/0.6814464516937733, Validation accuracy: 0.551
Epoch 5/20, Loss: 0.6654080653947497/0.685212817043066, Validation accuracy: 0.517
Epoch 6/20, Loss: 0.6544789246150425/0.6830159649252892, Validation accuracy: 0.5565
Epoch 7/20, Loss: 0.645477967602866/0.7099565863609314, Validation accuracy: 0.5435
Epoch 8/20, Loss: 0.6344556061048356/0.7199248187243938, Validation accuracy: 0.5325
Epoch 9/20, Loss: 0.6259254907804822/0.7050711736083031, Validation accuracy: 0.553
Epoch 10/20, Loss: 0.612234389971173/0.6998434625566006, Validation accuracy: 0.567
Epoch 11/20, Loss: 0.5975207289059957/0.7457321137189865, Validation accuracy: 0.5445
Epoch 12/20, Loss: 0.5866267539206005/0.7400864027440548, Validation accurac